# Conformal Reliability Assessment of PPG-Based ICU Arrhythmia Classification

Reproducibility notebook for the paper *"Conformal Reliability Assessment of PPG-Based
ICU Arrhythmia Classification under Class Imbalance and Patient Heterogeneity."*

**Main experiments are PPG-only.** The ECG-assisted analysis (Section 5.8) is optional
and enabled with the `--ecg-assisted` flag.

**Protocol note.** The calibration and test splits are never used for training, early
stopping, or checkpoint selection. The 70% training split is divided patient-disjoint
into inner-train / inner-val; early stopping uses inner-val only. Conformal thresholds
are computed only on the untouched calibration split.

This notebook drives the scripts in `src/`. Set `DATA_ROOT` in `configs/ppg_only.yaml`
first (see README).

In [ ]:
# Environment
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, "src")

## Step 1 — Build the expanded cohort and patient-disjoint split

In [ ]:
!python src/01_build_cohort.py --config configs/ppg_only.yaml

## Step 2 — Extract 14-dim PPG-only features (XGBoost baseline)

In [ ]:
!python src/02_extract_features.py --config configs/ppg_only.yaml

## Step 3 — Train the PPG-only XGBoost baseline
Early stopping uses the inner-validation subset (never calibration/test).

In [ ]:
!python src/03_train_xgboost.py --config configs/ppg_only.yaml

## Step 4 — Train the PPG-only InceptionTime model
Checkpoint selection uses the inner-validation subset only.

In [ ]:
!python src/04_train_inceptiontime.py --config configs/ppg_only.yaml

## Step 5 — Conformal evaluation
Produces LAC + APS (global / class-conditional), segment & patient-clustered bootstrap
CIs, SQI-stratum behaviour, SQI calibration variants, prevalence reweighting, and
per-patient coverage. CSVs are written to `outputs/reported_results/`.

In [ ]:
!python src/05_conformal.py --config configs/ppg_only.yaml

## Step 6 — Regenerate the paper figures (Fig. 1–5 and Appendix Fig. A.6)

In [ ]:
!python src/06_make_figures.py --config configs/ppg_only.yaml

## Optional — ECG-assisted sensitivity analysis (Section 5.8)
Adds the ECG channel; off by default. Run only if you want to reproduce Table 9.

In [ ]:
# !python src/02_extract_features.py    --config configs/ecg_assisted.yaml --ecg-assisted
# !python src/03_train_xgboost.py       --config configs/ecg_assisted.yaml --ecg-assisted
# !python src/04_train_inceptiontime.py --config configs/ecg_assisted.yaml --ecg-assisted
# !python src/05_conformal.py           --config configs/ecg_assisted.yaml --ecg-assisted

## Reported results
The CSVs under `outputs/reported_results/` contain the exact values used for every table
and figure in the paper (base classifier metrics, per-class coverage, bootstrap CIs, SQI
strata and variants, prevalence reweighting, ECG sensitivity, and split composition).

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob("outputs/reported_results/*.csv")):
    print("\n==", f, "==")
    print(pd.read_csv(f).to_string(index=False))